# Cox-Ross-Rubinstein Model: Methodological Note on Option Pricing

## Introduction

The Cox-Ross-Rubinstein (CRR) binomial model is a discrete-time framework for pricing derivatives, particularly options, on underlying assets whose prices evolve through a multiplicative binomial tree. Introduced by Cox, Ross, and Rubinstein in 1979, this model provides both a practical computational method and a bridge between discrete and continuous-time finance theory.

## Model Framework

### Binomial Price Evolution

At each time step $\Delta t$, the underlying asset price $S$ can move to one of two states:

- **Up move**: $S \to uS$ with probability $p$
- **Down move**: $S \to dS$ with probability $1-p$

where $u > 1$ and $0 < d < 1$ are the up and down multiplicative factors.

### Parameter Specification

The CRR model calibrates these parameters to match the continuous Black-Scholes dynamics:

$$u = e^{\sigma\sqrt{\Delta t}}$$

$$d = e^{-\sigma\sqrt{\Delta t}} = \frac{1}{u}$$

$$p = \frac{e^{r\Delta t} - d}{u - d}$$

where:
- $\sigma$ is the annualized volatility
- $r$ is the risk-free rate
- $\Delta t = T/N$ is the time step size (total maturity $T$ divided into $N$ steps)

The risk-neutral probability $p$ ensures no arbitrage and matches the expected return to the risk-free rate.

## Pricing Algorithm

### Backward Induction

Option valuation proceeds via backward induction through the binomial tree:

1. **Terminal payoff**: At maturity $T$, compute the option payoff at each terminal node.

2. **Recursive valuation**: At each earlier node and time step $t_i$, the option value is:
    $$V(S, t_i) = e^{-r\Delta t}\left[p \cdot V(uS, t_{i+1}) + (1-p) \cdot V(dS, t_{i+1})\right]$$

3. **Continuation**: Continue backward until reaching the initial node at $t=0$.

### Boundary Conditions

- **European options**: Exercise occurs only at maturity.
- **American options**: At each node, compare the intrinsic value with the continuation value, and take the maximum.

## Convergence Properties

### Convergence to Black-Scholes

As the number of steps $N \to \infty$ (and $\Delta t \to 0$), the binomial model converges to the continuous Black-Scholes formula. This convergence is guaranteed provided:

- The parameters $u$ and $d$ scale with $\sqrt{\Delta t}$
- The risk-neutral probability $p$ approaches $0.5$ as volatility and interest rates are held fixed

### Rate of Convergence

For European options, the error decreases approximately as $O(1/N)$. For American options, convergence is typically slower due to discrete exercise decisions.

## Advantages

- **Intuitive**: Easy to understand and implement; provides economic intuition via the tree structure.
- **Flexibility**: Naturally handles American and exotic options with path-dependent features.
- **Transparency**: The tree shows all possible price paths and decision nodes explicitly.
- **Calibration**: Parameters have direct interpretation in terms of volatility and risk-free rate.

## Limitations

- **Computational cost**: For $N$ steps and many options, storage and computation grow with $N^2$ in a recombinant tree.
- **Discrete approximation**: Volatility smile and skew cannot be modeled directly without adjusting parameters over time or across strikes.
- **Oscillation**: Numerical oscillations in convergence can occur for certain parameter combinations.
- **Convergence speed**: Slower than some other methods (e.g., PDE solvers) for plain-vanilla European options.

## Practical Implementation

### Recombinant vs. Non-Recombinant Trees

- **Recombinant**: After $n$ up moves and $m$ down moves, the price is $S_0 u^n d^m$, independent of path order. This reduces nodes from $2^N$ to $(N+1)(N+2)/2$.
- **Non-recombinant**: Used when state-dependent parameters or dividends are needed; requires more nodes and memory.

### Dividend Handling

For dividend-paying assets:
- **Continuous dividends**: Adjust the risk-neutral probability to account for dividend yield $q$:
  $$p = \frac{e^{(r-q)\Delta t} - d}{u - d}$$
- **Discrete dividends**: Subtract known dividend amounts from the stock price at ex-dividend dates.

## Numerical Considerations

1. **Number of steps**: Choose $N$ large enough for acceptable accuracy but manageable computation. $N = 100$ to $500$ is typical for simple options.

2. **Stability**: Ensure $0 < p < 1$ to avoid negative probabilities, which violates risk-neutral assumptions.

3. **Early exercise**: For American options, check at each node whether early exercise is optimal.

## Extensions

- **Multi-dimensional binomial**: Model correlated assets for basket or spread options.
- **Time-varying parameters**: Allow volatility or interest rates to depend on time or state.
- **Jump-diffusion binomial**: Incorporate jump risks in addition to diffusion.

## Summary

The Cox-Ross-Rubinstein binomial model is a foundational discrete-time framework for option pricing. It offers simplicity, flexibility, and intuitive appeal, making it invaluable for educational purposes and practical applications involving exotic or early-exercise features. While convergence to the Black-Scholes model is guaranteed, practitioners must balance accuracy and computational efficiency when choosing the number of tree steps. The CRR approach remains central to derivatives valuation and serves as a benchmark for understanding continuous-time models.

In [2]:
import numpy as np
import pandas as pd
import yfinance as yf

import matplotlib.pyplot as plt

ticker = 'AAPL'
df = yf.download(ticker, period='1y', interval='1d', progress=False).dropna()
close = df['Close']
S0 = close.iloc[-1]
returns = np.log(close / close.shift(1)).dropna()
sigma = returns.std() * np.sqrt(252)

r = 0.03
T = 0.25
K = S0 * 1.02
N = 100
u = np.exp(sigma * np.sqrt(dt))
d = 1 / u
p = (np.exp(r * dt) - d) / (u - d)

def build_crr_tree(S0, K, r, sigma, T, N):
    dt = T / N
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u
    p = (np.exp(r * dt) - d) / (u - d)

    asset_tree = [
        S0 * u ** np.arange(i, -1, -1) * d ** np.arange(0, i + 1)
        for i in range(N + 1)
    ]
    option_tree = [None] * (N + 1)
    option_tree[N] = np.maximum(asset_tree[N] - K, 0)

    for i in range(N - 1, -1, -1):
        option_tree[i] = np.exp(-r * dt) * (
            p * option_tree[i + 1][:-1] + (1 - p) * option_tree[i + 1][1:]
        )

    return asset_tree, option_tree

asset_tree, option_tree = build_crr_tree(S0, K, r, sigma, T, N)
call_price = option_tree[0][0]

central_indices = [i // 2 for i in range(N + 1)]
central_asset = [asset_tree[i][idx] for i, idx in enumerate(central_indices)]
central_option = [option_tree[i][idx] for i, idx in enumerate(central_indices)]

print(f'Ticker: {ticker}')
print(f'S0 = {S0:.2f}, K = {K:.2f}, T = {T:.2f} anni, sigma = {sigma:.2%}, r = {r:.2%}')
print(f'Prezzo call CRR (N={N}): {call_price:.2f}')

fig, axes = plt.subplots(2, 1, figsize=(12, 9))

axes[0].plot(close.index, close, label=f'{ticker} Close')
axes[0].set_title(f'{ticker} - Prezzi storici ultimi 12 mesi')
axes[0].set_ylabel('Prezzo')
axes[0].grid(True)
axes[0].legend()

axes[1].plot(asset_tree[N], np.maximum(asset_tree[N] - K, 0), label='Payoff a maturità')
axes[1].plot(central_asset, central_option, label='Valore call percorso centrale', marker='o')
axes[1].set_title('Albero binomiale CRR: payoff finale e percorso centrale del prezzo dell\'opzione')
axes[1].set_xlabel('Indice nodo / passo')
axes[1].set_ylabel('Prezzo')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

ValueError: Length of values (2) does not match length of index (1)

In [ ]:
def bin_matlab(u, T, t0, r, S0, k, c1, c2):
    import numpy as np
    dt = int(T - t0)
    d = 1.0 / u
    fs = np.exp(-r * dt)

    q = (fs - d) / (u - d)
    up_exps = np.arange(T, t0 - 1, -1)
    down_exps = np.arange(t0, T + 1)
    St = S0 * u ** up_exps * d ** down_exps

    D = np.zeros((dt + 1, dt + 1))

    if c1 == 1:
        H = np.maximum(St - k, 0)
    else:
        H = np.maximum(k - St, 0)
    D[:, -1] = H

    for i in range(T, t0 - 1, -1):
        idx = i - t0
        n = i - t0
        D[0:n + 1, idx] = fs * (q * D[0:n + 1, idx + 1] + (1 - q) * D[1:n + 2, idx + 1])
        if c2 == 1:
            VI = np.maximum(S0 * u ** np.arange(i, t0 - 1, -1) * d ** np.arange(t0, i + 1) - k, 0)
            D[0:n + 1, idx] = np.maximum(D[0:n + 1, idx], VI)

    D0 = D[0, 0]
    return D0, D


In [ ]:
# Usa i dati scaricati in precedenza e costruisce nuovamente l'albero CRR, poi plottiamo
dt = T / N
u = np.exp(sigma * np.sqrt(dt))
d = 1 / u

asset_tree, option_tree = build_crr_tree(S0, K, r, sigma, T, N)
call_price = option_tree[0][0]

central_indices = [i // 2 for i in range(N + 1)]
central_asset = [asset_tree[i][idx] for i, idx in enumerate(central_indices)]
central_option = [option_tree[i][idx] for i, idx in enumerate(central_indices)]

import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 1, figsize=(12, 9))

axes[0].plot(close.index, close, label=f'{ticker} Close')
axes[0].set_title(f'{ticker} - Prezzi storici ultimi 12 mesi')
axes[0].set_ylabel('Prezzo')
axes[0].grid(True)
axes[0].legend()

axes[1].plot(asset_tree[N], np.maximum(asset_tree[N] - K, 0), label='Payoff a maturità')
axes[1].plot(central_asset, central_option, label='Valore call percorso centrale', marker='o')
axes[1].set_title('Albero binomiale CRR: payoff finale e percorso centrale del prezzo dell\'opzione')
axes[1].set_xlabel('Indice nodo / passo')
axes[1].set_ylabel('Prezzo')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f'Prezzo call CRR (N={N}): {call_price:.2f}')